# Amazon Review Alignment: A100 Smoke -> Formal -> Baselines

?? Colab ????? A100 ?? Qwen3.5-2B ??? smoke test???????????SFT/DPO?Reward Model?PPO/GRPO ????????????????? A100 online profile??? baseline ??????????

## 1. ?? Drive ?????

????? checkpoint ??? Google Drive???????????? GitHub `main`??? notebook ?? Drive ??? clone ? fast-forward ????????

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path("/content/drive/MyDrive/amazon-review-alignment-workspace/repo")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "stash", "push", "-m", "colab-auto-stash-before-pull"],
            check=True,
        )
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. ????

????? Colab ?? **??? -> ??????**?????????????????????

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    cwd=REPO_DIR,
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")

## 3. ????? CLI helper

? Colab Secrets ??? `OPENAI_API_KEY`?`HF_TOKEN` ????????`DEEPSEEK_API_KEY` ???? DeepSeek baseline ????

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata
from packaging.version import Version

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path("/content/drive/MyDrive/amazon-review-alignment-workspace/repo")
os.chdir(REPO_DIR)

source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN", "DEEPSEEK_API_KEY"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

try:
    import importlib.metadata
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    print("Removing unused TorchAO:", torchao_version)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)


def training_stack_is_usable() -> bool:
    try:
        import amazon_review_alignment
        import bitsandbytes
        import peft
        import transformers
        import trl
    except (ImportError, ModuleNotFoundError):
        return False
    return Version(bitsandbytes.__version__) >= Version("0.46.1")


if not training_stack_is_usable():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
        cwd=REPO_DIR,
        check=True,
    )

import amazon_review_alignment
import bitsandbytes
import peft
import transformers
import trl


def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-u",
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command), flush=True)
    env = os.environ.copy()
    env["PYTHONPATH"] = source_dir + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONFAULTHANDLER"] = "1"
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(command, returncode, stdout="".join(output_lines), stderr=None)
    if check and returncode:
        print("\n===== LAST 200 LINES =====")
        print("".join(output_lines[-200:]))
        raise RuntimeError(f"Command failed with exit code {returncode}: " + " ".join(command))
    return result

print("Package:", Path(amazon_review_alignment.__file__).resolve())
print("Training stack:", transformers.__version__, trl.__version__, peft.__version__, bitsandbytes.__version__)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))
print("DeepSeek key loaded:", bool(os.getenv("DEEPSEEK_API_KEY")))

## 4. A100 ????

In [ ]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "a100" not in gpu_name.lower():
    raise RuntimeError("Expected A100, but Colab assigned: " + gpu_name)
if total_gib < 38:
    raise RuntimeError("Insufficient GPU memory for this A100 profile.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")

## 5. ?? smoke ??

Smoke ????? `rlhf_a100_online_v2.yaml` ?????? Qwen3.5-2B ? A100/BF16 ?????????????PPO/GRPO prompt ???????????????????????????? checkpoint?

In [ ]:
import json
from pathlib import Path

import yaml

from amazon_review_alignment.config import load_config

FORMAL_CONFIG = "configs/rlhf_a100_online_v2.yaml"
SMOKE_CONFIG_PATH = Path("/content/rlhf_a100_smoke.yaml")

formal = load_config(REPO_DIR / FORMAL_CONFIG)
formal.pop("_config_path", None)

old_root = "outputs/a100-qwen3.5-2b"
smoke_root = "outputs/a100-smoke-qwen3.5-2b"


def replace_output_paths(value):
    if isinstance(value, dict):
        return {key: replace_output_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_output_paths(item) for item in value]
    if isinstance(value, str):
        return value.replace(old_root, smoke_root)
    return value


smoke = replace_output_paths(formal)
smoke["project"]["output_dir"] = smoke_root
smoke["data"].update(
    {
        "sample_size": 60,
        "max_scanned_reviews": 20000,
        "rating_targets": {"1": 12, "2": 12, "3": 12, "4": 12, "5": 12},
        "splits": {"train": 42, "validation": 6, "test": 12},
    }
)
smoke["teacher"].update({"pilot_size": 5, "max_estimated_cost_usd": 1.0})
smoke["training"]["sft"]["max_steps"] = 1
smoke["training"]["dpo"]["max_steps"] = 1
smoke["rlhf"].update(
    {
        "human_calibration_samples": 0,
        "ai_reward_train_pairs": 4,
        "ai_reward_validation_pairs": 2,
        "ppo_prompt_count": 4,
    }
)
smoke["rlhf"]["reward"]["max_steps"] = 1
smoke["rlhf"]["ppo"].update({"total_episodes": 4, "gradient_accumulation_steps": 1, "save_steps": 1})
smoke["rlhf"]["grpo"].update({"prompt_count": 4, "num_generations": 2, "generation_batch_size": 2, "max_steps": 1, "save_steps": 1})
smoke["evaluation"].update(
    {
        "max_test_samples": 4,
        "variants": ["base", "sft", "dpo", "ppo", "grpo"],
        "judge_samples_per_pair": 4,
        "judge_pairs": [["sft", "dpo"], ["sft", "ppo"], ["sft", "grpo"], ["ppo", "grpo"]],
    }
)

SMOKE_CONFIG_PATH.write_text(yaml.safe_dump(smoke, sort_keys=False, allow_unicode=True), encoding="utf-8")

print("Smoke config:", SMOKE_CONFIG_PATH)
print("Formal config:", FORMAL_CONFIG)
print("Smoke output:", smoke["project"]["output_dir"])
print("Formal output:", formal["project"]["output_dir"])
print("Formal evaluation variants:", ", ".join(formal["evaluation"]["variants"]))

## 6. A100 smoke test

???????????????????? `teacher-batch` ?????????????????????? smoke ?????????

In [ ]:
import pandas as pd


def output_root_for(config_path: str | Path) -> Path:
    return Path(load_config(config_path)["project"]["output_dir"]).resolve()


def require_teacher_outputs(config_path: str | Path) -> None:
    root = output_root_for(config_path)
    train_preferences = root / "teacher" / "preferences_train.jsonl"
    validation_preferences = root / "teacher" / "preferences_validation.jsonl"
    if not train_preferences.exists() or not validation_preferences.exists():
        raise RuntimeError("Teacher batch is not complete yet. Re-run teacher-batch later, then rerun this cell.")
    print("Teacher train rows:", sum(1 for line in train_preferences.open(encoding="utf-8") if line.strip()))
    print("Teacher validation rows:", sum(1 for line in validation_preferences.open(encoding="utf-8") if line.strip()))


SMOKE_CONFIG = str(SMOKE_CONFIG_PATH)
subprocess.run([sys.executable, "-m", "pytest"], cwd=REPO_DIR, check=True)
cli("prepare-data", "--config", SMOKE_CONFIG)
cli("evaluate", "--config", SMOKE_CONFIG, "--variants", "base", "--force-inference")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", SMOKE_CONFIG)
cli("teacher-batch", "--config", SMOKE_CONFIG)
require_teacher_outputs(SMOKE_CONFIG)

cli("train-sft", "--config", SMOKE_CONFIG)
cli("merge-sft", "--config", SMOKE_CONFIG)
cli("train-dpo", "--config", SMOKE_CONFIG)
cli("build-rlhf-data", "--config", SMOKE_CONFIG)
cli("train-reward", "--config", SMOKE_CONFIG)
cli("train-ppo", "--config", SMOKE_CONFIG)
cli("train-grpo", "--config", SMOKE_CONFIG)
cli("evaluate", "--config", SMOKE_CONFIG, "--variants", "base", "sft", "dpo", "ppo", "grpo", "--force-inference")
cli("build-report", "--config", SMOKE_CONFIG)

display(pd.read_csv(output_root_for(SMOKE_CONFIG) / "evaluation" / "metrics.csv"))
print("Smoke report:", output_root_for(SMOKE_CONFIG) / "evaluation" / "report.md")

## 7. ????????????

?????? `configs/rlhf_a100_online_v2.yaml`??? DPO v2?1,024 ? PPO/GRPO shared prompts??? baseline ?????Batch ???????? GPU???????????

In [ ]:
FORMAL_ROOT = output_root_for(FORMAL_CONFIG)
print("Formal root:", FORMAL_ROOT)

cli("prepare-data", "--config", FORMAL_CONFIG)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", FORMAL_CONFIG)
cli("teacher-batch", "--config", FORMAL_CONFIG)
require_teacher_outputs(FORMAL_CONFIG)

## 8. ?????SFT?DPO?Reward Model?PPO?GRPO

??????? Drive ???????????????????????????????

In [ ]:
cli("train-sft", "--config", FORMAL_CONFIG)
cli("merge-sft", "--config", FORMAL_CONFIG)
cli("train-dpo", "--config", FORMAL_CONFIG)
cli("build-rlhf-data", "--config", FORMAL_CONFIG)
cli("train-reward", "--config", FORMAL_CONFIG)
cli("train-ppo", "--config", FORMAL_CONFIG)
cli("train-grpo", "--config", FORMAL_CONFIG)

manifest_path = FORMAL_ROOT / "rlhf" / "data_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["ppo_grpo_shared_prompt_ids"] is True

## 9. ????????

In [ ]:
TRAINED_VARIANTS = ["base", "sft", "dpo", "ppo", "grpo"]
cli("evaluate", "--config", FORMAL_CONFIG, "--variants", *TRAINED_VARIANTS)
cli("build-report", "--config", FORMAL_CONFIG)

display(pd.read_csv(FORMAL_ROOT / "evaluation" / "metrics.csv"))
print("Report:", FORMAL_ROOT / "evaluation" / "report.md")

## 10. Baselines ???????

????????/?? baseline????? `DEEPSEEK_API_KEY`?????? DeepSeek few-shot baseline????????? metrics ??

In [ ]:
BASELINE_VARIANTS = [
    "qwen35_2b_fewshot",
    "phi4_mini_fewshot",
    "nlptown_template",
]
if os.getenv("DEEPSEEK_API_KEY"):
    BASELINE_VARIANTS.append("deepseek_v4_pro_fewshot")
else:
    print("Skipping deepseek_v4_pro_fewshot because DEEPSEEK_API_KEY is not set.")

for variant in BASELINE_VARIANTS:
    cli("inference", "--config", FORMAL_CONFIG, "--variant", variant)

ALL_EVAL_VARIANTS = [*TRAINED_VARIANTS, *BASELINE_VARIANTS]
cli("evaluate", "--config", FORMAL_CONFIG, "--variants", *ALL_EVAL_VARIANTS)
cli("build-report", "--config", FORMAL_CONFIG)

metrics = pd.read_csv(FORMAL_ROOT / "evaluation" / "metrics.csv")
display(metrics)
print("Variants evaluated:", ", ".join(ALL_EVAL_VARIANTS))
print("Report:", FORMAL_ROOT / "evaluation" / "report.md")

## 11. ???AI ??

????? OpenAI judge???? API ???????? predictions??? `--force-inference`?

In [ ]:
RUN_AI_JUDGE = False
AI_JUDGE_SAMPLES_PER_PAIR = 100

if RUN_AI_JUDGE:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
    cli(
        "evaluate",
        "--config",
        FORMAL_CONFIG,
        "--variants",
        *ALL_EVAL_VARIANTS,
        "--llm-judge",
        "--judge-samples-per-pair",
        str(AI_JUDGE_SAMPLES_PER_PAIR),
    )
    cli("build-report", "--config", FORMAL_CONFIG)
    display(pd.read_csv(FORMAL_ROOT / "evaluation" / "judge_pairwise_summary.csv"))
    print("Report:", FORMAL_ROOT / "evaluation" / "report.md")
else:
    print("Set RUN_AI_JUDGE = True to run blinded pairwise judging.")